In [1]:
import pandas as pd
import joblib
from pyprojroot import here
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score
from xgboost import XGBClassifier
from features import fit_encoders, transform_features

# Leakage Comparison — Three Models

Three **independent** models, each trained on a separate, standalone feature set,
to compare how leakage of differing severity affects the score. They are not built
on top of each other — each uses only its own features, on the identical December
test set and identical XGBoost config:

1. **Leaky** — features that *define* the target (the actual departure delay).
2. **Moderately leaky** — post-departure consequences that hint at the target but don't
   define it (arrival delay, taxi-out time).
3. **Clean (deployable)** — only schedule data and train-only historical delay
   rates, all available 2 hours before departure.

The contrast across the three shows how much apparent performance comes from leakage
of each severity, and confirms the clean model's modest score is the honest one.

**The leaky and less-leaky models are demonstrations only. Neither is deployed.**

In [2]:
# Full data still carries the leaky columns; rebuild the same temporal split
df = pd.read_parquet(here("data/processed/flights.parquet"))
df = df[(df["Cancelled"] == 0) & (df["Diverted"] == 0)].copy()
train = df[df["Month"].isin([8, 9, 10])].copy()
test  = df[df["Month"] == 12].copy()

target = "DepDel15"
y_train, y_test = train[target], test[target]

XGB = dict(n_estimators=2000, max_depth=6, learning_rate=0.1,
           subsample=0.8, colsample_bytree=0.8,
           eval_metric="aucpr", random_state=42)

def run(name, X_train, X_test):
    m = XGBClassifier(**XGB).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {"model": name,
            "accuracy": round(accuracy_score(y_test, pred), 4),
            "PR_AUC":  round(average_precision_score(y_test, p), 4),
            "ROC_AUC": round(roc_auc_score(y_test, p), 4)}

rows = []

# 1) Fully leaky — columns that define the target
full = ["DepDelay", "DepDelayMinutes"]
rows.append(run("1. Fully leaky", train[full], test[full]))

# 2) Moderately leaky — post-departure consequences, not the definition
mod = ["ArrDelay", "TaxiOut"]
rows.append(run("2. Moderately leaky", train[mod], test[mod]))

# 3) Clean — schedule + train-only historical delay rates
enc = fit_encoders(train)
rows.append(run("3. Clean (deployable)",
                transform_features(train, enc),
                transform_features(test, enc)))

comp = pd.DataFrame(rows)
print(comp.to_string(index=False))
print(f"\nBaseline accuracy (predict all on-time): {1 - y_test.mean():.3f}")
print(f"Baseline PR-AUC (December delay rate):   {y_test.mean():.3f}")

                model  accuracy  PR_AUC  ROC_AUC
       1. Fully leaky    1.0000  1.0000   1.0000
  2. Moderately leaky    0.9304  0.9172   0.9679
3. Clean (deployable)    0.7834  0.3249   0.6523

Baseline accuracy (predict all on-time): 0.784
Baseline PR-AUC (December delay rate):   0.216


## Results

| model | accuracy | PR-AUC | ROC-AUC |
|-------|----------|--------|---------|
| 1. Fully leaky | **1.0000** | **1.0000** | **1.0000** |
| 2. Moderately leaky | **0.9304** | **0.9172** | **0.9679** |
| 3. Clean (deployable) | **0.7834** | **0.3249** | **0.6523** |

Baselines: 0.784 accuracy / 0.216 PR-AUC (no-skill, December).

## What each leak fakes in isolation
Each model is trained on one standalone feature set, so its score measures how much
that severity of leak — on its own — can manufacture.

- **Leaky** — near-perfect, because `DepDelay` *is* the target. A 99% score that
  needs the answer as an input is the textbook signature of leakage, not skill.
- **Moderately leaky** — the dangerous one: ~0.9 ROC-AUC without obviously cheating.
  Arrival delay and taxi-out are measured *after* departure, so they'd never exist
  at prediction time — yet a careless pipeline would ship this and it would collapse
  in production. Moderate leaks are hard to catch precisely because the score looks
  plausible.
- **Clean** — scores lowest, but is the only honest, deployable model: every feature
  is available 2 hours before departure.

Read each score on its own terms: how far it sits above the clean model is how much
that single class of leakage can fabricate from information that wouldn't exist at
prediction time.

**Why accuracy is the wrong metric (the clean row says it all):** the clean model's accuracy (~0.78) is *no better than* blindly predicting "always on-time" (0.784) — yet its PR-AUC (0.3249) clears the 0.216 no-skill floor by half again. Accuracy is blind to the genuine ranking skill PR-AUC exposes. That single row is the entire reason this project is judged on PR-AUC, not accuracy.
